In [0]:
%run ./00_config

In [0]:
%run ./02_core_functions

# 03_main
Orchestrates the full pipeline for every active control row: **copy** the zip files from
`zip_source_path` to their date-partitioned `zip_tgt_path`, then **unzip** the files into
`unzip_tgt_path`. Both stages are idempotent — re-running this notebook checks `log_table` and
only (re)processes files that are new or previously failed; files already logged as `SUCCESS`
are left untouched.

### Run setup

In [0]:
run_id = str(uuid.uuid4())
control_entries = get_active_control_entries()

print(f"run_id = {run_id}")
print(f"Processing {len(control_entries)} active control row(s)")

all_log_records = []

### Stage 1 — Copy
For every active control row, list zip files at `zip_source_path`, keep only the ones that
match the load type, then drop any that were **already copied successfully in an earlier run**
(checked against `log_table`):
- `is_full_load = 1` -> all zip files found match (unchanged behavior).
- `is_incremental_load = 1` -> only files whose filename date token matches **today's date**
  (`yyyyMMdd`, system date) match. If none match at all, a `FAILED` log record is written for
  that control row.
- Of the matching files, only ones that are new or whose last COPY attempt did **not** succeed
  (`FAILED`/`SKIPPED`/never attempted) are actually copied this run.

Selected files are copied into the matching `yyyyMMdd` folder under `zip_tgt_path`.

In [0]:
copy_status = get_latest_stage_status("COPY")

for ctl in control_entries:
    print(f"--- s_no={ctl['s_no']} | source={ctl['zip_source']} | {ctl['zip_source_path']} ---")

    all_zip_files = list_zip_files(ctl["zip_source_path"])
    load_filtered_files = filter_zip_files_by_load_type(
        all_zip_files,
        is_full_load=ctl["is_full_load"],
        is_incremental_load=ctl["is_incremental_load"],
    )

    if ctl["is_incremental_load"] == 1 and len(load_filtered_files) == 0:
        today_token = datetime.now().strftime("%Y%m%d")
        now = datetime.now()
        failure_record = {
            "s_no": ctl["s_no"],
            "stage": "COPY",
            "source": ctl["zip_source"],
            "file_name": None,
            "source_path": ctl["zip_source_path"],
            "target_path": None,
            "status": "FAILED",
            "message": f"Incremental load: no zip file found for today's date {today_token}",
            "start_time": now,
            "end_time": now,
        }
        all_log_records.append(failure_record)
        print(f"  [COPY:FAILED] {failure_record['message']}")
        continue

    # Skip files whose latest COPY attempt already succeeded in a previous run
    zip_files = [
        f for f in load_filtered_files
        if copy_status.get((ctl["s_no"], f.name)) != "SUCCESS"
    ]
    already_done = len(load_filtered_files) - len(zip_files)

    print(f"Found {len(all_zip_files)} zip file(s) at source, {len(load_filtered_files)} match load type "
          f"(is_full_load={ctl['is_full_load']}, is_incremental_load={ctl['is_incremental_load']}), "
          f"{already_done} already copied successfully -> {len(zip_files)} to copy this run")

    if not zip_files:
        print("  Nothing new to copy for this control row (already up to date)")
        continue

    for file_info in zip_files:
        record = copy_zip_file(
            file_info=file_info,
            target_base_path=ctl["zip_tgt_path"],
            s_no=ctl["s_no"],
            source=ctl["zip_source"],
        )
        all_log_records.append(record)
        print(f"  [COPY:{record['status']}] {record['file_name']}: {record['message']}")

### Stage 2 — Unzip
Unzip candidates are the union of:
- files copied successfully **in this run**, and
- files copied successfully in an **earlier** run that were never successfully unzipped
  (`get_pending_unzip_from_history`).

Any file whose latest UNZIP attempt already succeeded is skipped, so re-running the notebook
never re-extracts files that are already done.

In [0]:
# s_no -> unzip_tgt_path, for routing each copied file to the right destination
unzip_targets = {ctl["s_no"]: ctl["unzip_tgt_path"] for ctl in control_entries}

copied_this_run = [
    r for r in all_log_records
    if r["stage"] == "COPY" and r["status"] == "SUCCESS"
]

pending_from_history = get_pending_unzip_from_history()
seen_keys = {(r["s_no"], r["file_name"]) for r in copied_this_run}
history_only = [r for r in pending_from_history if (r["s_no"], r["file_name"]) not in seen_keys]

# Safety net: drop anything whose latest UNZIP attempt already succeeded
unzip_status = get_latest_stage_status("UNZIP")
unzip_candidates = [
    r for r in (copied_this_run + history_only)
    if unzip_status.get((r["s_no"], r["file_name"])) != "SUCCESS"
]

print(f"{len(copied_this_run)} file(s) copied successfully this run, "
      f"{len(history_only)} pending from earlier run(s) -> "
      f"{len(unzip_candidates)} file(s) queued for unzip")

for copy_record in unzip_candidates:
    unzip_base = unzip_targets.get(copy_record["s_no"])
    if not unzip_base:
        print(f"  [UNZIP:SKIPPED] {copy_record['file_name']}: no unzip_tgt_path for s_no={copy_record['s_no']}")
        continue

    unzip_record = unzip_zip_file(
        copy_record=copy_record,
        unzip_base_path=unzip_base,
    )
    all_log_records.append(unzip_record)
    print(f"  [UNZIP:{unzip_record['status']}] {unzip_record['file_name']}: {unzip_record['message']}")

### Write Logs 
Both `COPY` and `UNZIP` records for this `run_id` are written in the same log table.

In [0]:
write_logs(all_log_records, run_id)
print(f"Logged {len(all_log_records)} record(s) to {log_table}")

### Run summary

In [0]:
%sql
SELECT * FROM iics.casemix.zip_files_execution_log_test
ORDER BY start_time